# VQE（変分量子固有値ソルバー）

量子回路で基底状態を近似的に作り、基底エネルギー$E_0$を求める方法。量子コンピュータで状態を作ってエネルギーを測り、古典コンピュータでパラメータを最適化する。

各模型への適用（アンザッツの選び方、測定のしかた、手計算）は、`models/<模型>/04_vqe.ipynb` にまとめてある。

## 1. 考え方

- **変分原理**

  任意の規格化された状態$\ket{\psi}$について

  \begin{equation}
  E(\psi) = \bra{\psi}H\ket{\psi} \ge E_0
  \end{equation}

  が成り立ち、等号は$\ket{\psi}$が基底状態のときに限る。そこでパラメータ$\boldsymbol{\theta}$を持つ量子回路で状態$\ket{\psi(\boldsymbol{\theta})}$を作り、$E(\boldsymbol{\theta})$を最小化すれば$E_0$に近づく。これがVQEである。

- **アルゴリズムの流れ**

  1. 量子コンピュータで$\ket{\psi(\boldsymbol{\theta})}$を作り、$E(\boldsymbol{\theta})$を測定する
  2. 古典コンピュータの最適化手法で$\boldsymbol{\theta}$を更新する
  3. $E(\boldsymbol{\theta})$が収束するまで1, 2を繰り返す

## 2. エネルギーの測定

ハミルトニアンをPauli演算子の積の和$H = \sum_a h_a P_a$で書くと、$E(\boldsymbol{\theta}) = \sum_a h_a\langle P_a\rangle$である。各$\langle P_a\rangle$は、Trotter分解のノートブック（[03_trotter.ipynb](03_trotter.ipynb)の4節）と同じ基底変換をしてから測定すれば求まる。同時に対角化できる（同じ基底で測れる）項をまとめて測ると、測定の種類を減らせる。

## 3. アンザッツ（回路の形）

パラメータ付きの回路$\ket{\psi(\boldsymbol{\theta})}$の形をアンザッツと呼ぶ。どう選ぶかで結果が大きく変わる。

- **Hardware-efficient ansatz**：実機で実行しやすい1量子ビット回転と、隣どうしのエンタングルゲートを交互に重ねる。汎用的だが、物理的な構造を使わないので、必要なパラメータが多くなりやすい。
- **Hamiltonian Variational Ansatz（HVA）**：$H = \sum_k H_k$の各部分の指数関数$e^{-i\theta_k H_k}$を交互に重ねる。Trotter分解の1ステップで、回転角を自由なパラメータにしたものと同じ形である。量子最適化のQAOAも同じ構造を持つ。ハミルトニアンの対称性を自然に保つ。
- **UCC（unitary coupled cluster）**：量子化学で使われる、電子の励起の形をもとにしたアンザッツ。

表現力が足りないと厳密な基底状態に届かず、逆にパラメータが多すぎると最適化が難しくなる。

## 4. 最適化と注意点

- **最適化手法**：勾配を使わない方法（COBYLA、Nelder-Meadなど）と、勾配を使う方法（BFGS、Adamなど）がある。実機では、回路のパラメータを$\pm\pi/2$ずらした2回の測定から勾配を求める**パラメータシフト則**が使える。
- **局所解**：エネルギーの地形には局所的な最小値が多く、初期値によって結果が変わる。層を1つずつ増やし、前の層の最適値を初期値に使う（ウォームスタート）と安定しやすい。
- **Barren plateau**：量子ビット数や回路の深さが大きいと、ランダムな初期値では勾配が指数関数的に小さくなり、最適化が進まなくなることがある。
- **変分原理による保証**：得られるエネルギーは必ず$E_0$以上なので、値が小さいほど良い近似である。ただし、どれだけ$E_0$に近いかはVQEだけではわからない。厳密解など他の手法と比べる必要がある。

---
**前へ**: [03 Trotter分解](03_trotter.ipynb)　｜　**次へ**: [05 TEBD](05_tebd.ipynb)